In [ ]:
import json
from pathlib import Path
from collections import Counter

base = Path(
    "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017"
)
annotation_dir = base / "annotations"

split_ids = {}
split_filenames = {}

for split in ["train2017", "val2017"]:
    json_path = annotation_dir / f"captions_{split}.json"
    image_dir = base / split

    with json_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    images = data["images"]
    captions = data["annotations"]

    image_ids = [img["id"] for img in images]
    filenames = [img["file_name"] for img in images]
    id_set = set(image_ids)

    caption_image_ids = {ann["image_id"] for ann in captions}
    orphan_captions = caption_image_ids - id_set

    caption_counts = Counter(
        ann["image_id"] for ann in captions
    )

    actual_files = {
        p.name for p in image_dir.iterdir() if p.is_file()
    }
    missing_files = set(filenames) - actual_files

    split_ids[split] = id_set
    split_filenames[split] = set(filenames)

    print(f"\n--- {split} ---")
    print("Images in JSON:", len(images))
    print("Unique image IDs:", len(id_set))
    print("Duplicate image IDs:", len(image_ids) - len(id_set))
    print("Caption records:", len(captions))
    print("Captions referencing unknown image IDs:", len(orphan_captions))
    print("Images missing from directory:", len(missing_files))
    print("Minimum captions per image:", min(caption_counts.values(), default=0))
    print("Maximum captions per image:", max(caption_counts.values(), default=0))

print("\n--- TRAIN / VALIDATION SEPARATION ---")
print("Overlapping image IDs:", len(split_ids["train2017"] & split_ids["val2017"]))
print("Overlapping filenames:", len(split_filenames["train2017"] & split_filenames["val2017"]))

In [ ]:
import json
import random
from pathlib import Path

# Read-only COCO dataset
COCO_ROOT = Path(
    "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017"
)

ANNOTATIONS_FILE = COCO_ROOT / "annotations" / "captions_train2017.json"

# Writable output directory
OUTPUT_DIR = Path("/kaggle/working/rag-artifacts/manifests")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 20261027
NUM_TRAIN_IMAGES = 10_000

SELECTION_FILE = OUTPUT_DIR / "train_subset_10000.json"
FILENAMES_FILE = OUTPUT_DIR / "train_subset_10000_filenames.txt"
METADATA_FILE = OUTPUT_DIR / "captions_metadata_10000.json"

# 1. Validate paths
if not ANNOTATIONS_FILE.is_file():
    raise FileNotFoundError(
        f"Training captions not found: {ANNOTATIONS_FILE}"
    )

# 2. Load the original COCO annotations
with ANNOTATIONS_FILE.open("r", encoding="utf-8") as f:
    data = json.load(f)

images = data["images"]
annotations = data["annotations"]

if len(images) < NUM_TRAIN_IMAGES:
    raise ValueError(
        f"Expected at least {NUM_TRAIN_IMAGES} images, found {len(images)}"
    )

# 3. Select the same deterministic subset as the original script
images = sorted(images, key=lambda item: int(item["id"]))

rng = random.Random(SEED)
selected = sorted(
    rng.sample(images, NUM_TRAIN_IMAGES),
    key=lambda item: int(item["id"])
)

selected_ids = {int(image["id"]) for image in selected}

# 4. Map captions to their selected images
captions_by_id = {image_id: [] for image_id in selected_ids}

for annotation in annotations:
    image_id = int(annotation["image_id"])

    if image_id in captions_by_id:
        captions_by_id[image_id].append(annotation["caption"])

# 5. Build the shared manifest
records = []

for image in selected:
    image_id = int(image["id"])
    filename = image["file_name"]

    records.append({
        "id": image_id,
        "file_name": filename,
        "captions": captions_by_id[image_id],
    })

# 6. Save the manifest
with SELECTION_FILE.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "seed": SEED,
            "count": len(records),
            "source": str(ANNOTATIONS_FILE),
            "images": records,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

with FILENAMES_FILE.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(record["file_name"] + "\n")

with METADATA_FILE.open("w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print("Selection created successfully.")
print("Selected images:", len(records))
print("Unique image IDs:", len(selected_ids))
print("Captions mapped:", sum(len(r["captions"]) for r in records))
print("Images with no captions:", sum(not r["captions"] for r in records))
print("Selection file:", SELECTION_FILE)
print("Metadata file:", METADATA_FILE)

In [ ]:
TRAIN_IMAGES_DIR = COCO_ROOT / "train2017"

missing = [
    record["file_name"]
    for record in records
    if not (TRAIN_IMAGES_DIR / record["file_name"]).is_file()
]

print("Expected images:", len(records))
print("Missing images:", len(missing))
print("First missing filenames:", missing[:10])

assert not missing, "Some selected images are missing."
print("PASS: every selected image exists.")

In [ ]:
import json
from pathlib import Path
from PIL import Image

REPORT_DIR = Path("/kaggle/working/rag-artifacts/reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_FILE = REPORT_DIR / "image_integrity_report.json"

corrupted = []
valid_count = 0

for i, record in enumerate(records, start=1):
    image_path = TRAIN_IMAGES_DIR / record["file_name"]

    try:
        # Check the image file structure.
        with Image.open(image_path) as img:
            img.verify()

        # Reopen and fully decode the image pixels.
        with Image.open(image_path) as img:
            img.load()

        valid_count += 1

    except Exception as error:
        corrupted.append({
            "file_name": record["file_name"],
            "error": str(error)
        })

    if i % 1000 == 0:
        print(f"Checked {i}/{len(records)} images")

report = {
    "expected_images": len(records),
    "valid_images": valid_count,
    "corrupted_images_count": len(corrupted),
    "corrupted_images": corrupted,
    "status": "PASS" if not corrupted and valid_count == len(records) else "FAIL"
}

with REPORT_FILE.open("w", encoding="utf-8") as file:
    json.dump(report, file, indent=2)

print("\n--- IMAGE INTEGRITY AUDIT ---")
print("Expected images:", report["expected_images"])
print("Valid images:", report["valid_images"])
print("Corrupted images:", report["corrupted_images_count"])
print("Status:", report["status"])
print("Report saved to:", REPORT_FILE)

In [ ]:
from pathlib import Path

repo = Path("/kaggle/working/RAG-for-generating-image-captioning")

print("Repository folder exists:", repo.exists())

if repo.exists():
    print("Contents:")
    for item in list(repo.iterdir())[:20]:
        print(item.name)